In [2]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
from scipy.spatial import cKDTree
import seaborn as sns
from utils import construct_path, read_feather_gcs
pd.set_option('display.max_rows', 200)

In [ ]:
### LOAD DATA
DATASET = "malecns_09"
SUBSET_NAME = "optic"
DATA_PATH = (Path.cwd().parent/"connectome_data")

pin_data  = 'pin_data'
out_dir = Path(Path.cwd().parent/"connectome_data/processed")

dataset_base = DATASET.split("_")[0]  # "malecns"

if SUBSET_NAME is not None:
    edgelist_path = (
        f"{DATA_PATH}/{dataset_base}/"
        f"{dataset_base}_{SUBSET_NAME}_{DATASET}_{SUBSET_NAME}_simple_edgelist.feather"
    )

    synapses_path = (
        f"{DATA_PATH}/{dataset_base}/"
        f"{dataset_base}_{SUBSET_NAME}_{DATASET}_{SUBSET_NAME}_synapses.feather"
    )
    pin_data_path = (
        f"{DATA_PATH}/{dataset_base}/")
    print(f"Using subset: {SUBSET_NAME}")
else:
    edgelist_path = construct_path(DATA_PATH, DATASET, "edgelist_simple")
    synapses_path = construct_path(DATA_PATH, DATASET, "synapses")
    print("Using full dataset")

meta_path = construct_path(str(DATA_PATH), DATASET, "meta")
print(f"Meta path: {meta_path}")
print(f"Edgelist path: {edgelist_path}")
print(f"Synapses path: {synapses_path}")

#Column pins from Nern et al 2025. Nature
#NOTE: !!!!!!
#Coordinate systems
# X Y Z
# Data from neuprint uses x y z coordinates. Here is a place to describe that coordinate system.

# Hex1 Hex2
# Omatidia in the Drosophila compound eye are arranged in what resembles a hexagonal grid. 
# Each individual omatidia can be uniqueliy identified along the 3 hexagonal axis hex1, hex2, and hex3. 
# In fact, any two of these coordinates is enough to identify the location. 
# The hex1 hex2 coordinates used in the neuprint database and in our analysis are positive numbers. 
# The origin of the coordinate system is arbitrarily located outside of the eyemap.

# P Q V
# A biologically more relevant coordinate system uses the center of the eye as the origin 
# of the P and Q coordinate system. We define the hex1 hex2 coordintes [18 19] as our origin for P Q.

# The H and V-axis use the same origin with V representing the virtual hex3. 
# The H axis is somewhat parallel to the perceived horizon.

LO = pd.read_csv(pin_data_path + "pin_data/LO_pindata.csv")
LOP = pd.read_csv(pin_data_path + "pin_data/LOP_pindata.csv")
ME = pd.read_csv(pin_data_path + "pin_data/ME_pindata.csv")
## Load nodes, edges and raw synapse data (Needed for CT1)
meta = read_feather_gcs(meta_path, gcs_fs=None)
meta["malecns_09_id"] = meta["malecns_09_id"].astype("int64")

nodes_df = meta.query("region == 'optic_lobe'").copy().reset_index(drop=True)
cols_to_keep= ['malecns_09_id', 'side', 'cell_type','neurotransmitter_predicted', 'neurotransmitter_score',
                'hex_1', 'hex_2',
                ]
nodes_df.rename(columns={"optic_lobe_hex_1": "hex_1",
                               "optic_lobe_hex_2": "hex_2"
                               }, inplace=True)
nodes_df = nodes_df[cols_to_keep].reset_index(drop=True)
# nodes_df = nodes_df.query('side == "right"').copy().reset_index(drop=True) #Only right side of the brain, to match the pin data

edgelist = read_feather_gcs(edgelist_path, gcs_fs=None)
dataset = ds.dataset(synapses_path, format="ipc")
print("Raw synapse dataset:")
dataset.head(10).to_pandas() #For some reason prepost is only 0. prepost	Link type (0 = presynapse, 1 = postsynapse)

Using subset: optic
Meta path: /home/guardomayas/NUIN/sjbcs/male_cns_var/connectome_data/malecns/malecns_09_meta.feather
Edgelist path: /home/guardomayas/NUIN/sjbcs/male_cns_var/connectome_data/malecns/malecns_optic_malecns_09_optic_simple_edgelist.feather
Synapses path: /home/guardomayas/NUIN/sjbcs/male_cns_var/connectome_data/malecns/malecns_optic_malecns_09_optic_synapses.feather
Loaded 165,114 rows
Loaded 6,629,951 rows
Raw synapse dataset:


,x,y,z,confidence,pre,post,neuropil,prepost,side,id,...,gaba,glutamate,dopamine,serotonin,octopamine,histamine,syn_top_p,syn_top_nt,pre_label,post_label
0,55170,27803,11260,0.815,512245,11517,AL,0.0,left,4.952206e+16,...,0.069234,0.848134,0.046629,0.015263,0.000395,0.000001,0.848134,glutamate,dendrite,None
1,55170,27803,11260,0.815,512245,744475431,AL,0.0,left,4.952206e+16,...,0.069234,0.848134,0.046629,0.015263,0.000395,0.000001,0.848134,glutamate,dendrite,None
2,54953,25832,11361,0.849,308347014,512245,AL,0.0,left,4.996626e+16,...,0.485572,0.000088,0.001202,0.003982,0.000491,0.000000,0.508665,acetylcholine,None,None
3,54998,25894,11404,0.853,512245,308347014,AL,0.0,left,5.015538e+16,...,0.023936,0.550873,0.000138,0.418448,0.000373,0.000068,0.550873,glutamate,dendrite,None
4,54998,25894,11404,0.853,512245,452952638,AL,0.0,left,5.015538e+16,...,0.023936,0.550873,0.000138,0.418448,0.000373,0.000068,0.550873,glutamate,dendrite,None
5,54998,25894,11404,0.853,512245,662169016,AL,0.0,left,5.015538e+16,...,0.023936,0.550873,0.000138,0.418448,0.000373,0.000068,0.550873,glutamate,dendrite,None
6,54998,25894,11404,0.853,512245,563901,AL,0.0,left,5.015538e+16,...,0.023936,0.550873,0.000138,0.418448,0.000373,0.000068,0.550873,glutamate,dendrite,None
7,54998,25894,11404,0.853,512245,180664781,AL,0.0,left,5.015538e+16,...,0.023936,0.550873,0.000138,0.418448,0.000373,0.000068,0.550873,glutamate,dendrite,None
8,52112,16932,11686,0.972,16334,10286,CentralBrain-unspecified,0.0,None,5.139561e+16,...,0.037035,0.009379,0.008058,0.005972,0.004652,0.000008,0.934896,acetylcholine,axon,None
9,51951,16908,11698,0.978,515811,10286,CentralBrain-unspecified,0.0,None,5.144838e+16,...,0.056080,0.011280,0.016250,0.021685,0.002546,0.000068,0.892091,acetylcholine,axon,None


In [3]:
## Add p q coordinates 
nodes_df["hex_1"] = pd.to_numeric(
    nodes_df["hex_1"], errors="coerce"
)
nodes_df["hex_2"] = pd.to_numeric(
    nodes_df["hex_2"], errors="coerce"
)
nodes_df["p"] = nodes_df["hex_1"] - 18
nodes_df["q"] = nodes_df["hex_2"] - 19
nodes_df.head()

columnar_nodes = nodes_df.query(
    "hex_1.notna() and hex_2.notna()"
).copy().reset_index(drop=True)
columnar_nodes

,malecns_09_id,side,cell_type,neurotransmitter_predicted,neurotransmitter_score,hex_1,hex_2,p,q
0,100043,right,C2,gaba,0.528316,24.0,17.0,6.0,-2.0
1,100062,right,C2,unclear,0.398571,24.0,35.0,6.0,16.0
2,100134,right,C2,unclear,0.405942,22.0,31.0,4.0,12.0
3,100156,right,C2,unclear,0.467389,28.0,38.0,10.0,19.0
4,100229,right,C2,unclear,0.581083,27.0,13.0,9.0,-6.0
...,...,...,...,...,...,...,...,...,...
23715,99591,left,Tm9,acetylcholine,0.974427,17.0,6.0,-1.0,-13.0
23716,99729,left,Tm9,acetylcholine,0.959740,12.0,13.0,-6.0,-6.0
23717,99798,right,Tm9,acetylcholine,0.959936,10.0,15.0,-8.0,-4.0
23718,99825,right,Tm9,acetylcholine,0.974427,10.0,17.0,-8.0,-2.0


## CT1

In [4]:
nodes_df[nodes_df['cell_type'] == 'CT1']

,malecns_09_id,side,cell_type,neurotransmitter_predicted,neurotransmitter_score,hex_1,hex_2,p,q
3536,10009,left,CT1,gaba,0.860479,NaN,NaN,NaN,NaN
3537,10157,right,CT1,gaba,0.871400,NaN,NaN,NaN,NaN


In [5]:
ct1_info = meta.query("cell_type == 'CT1'")
ct1_ids = ct1_info['malecns_09_id'].unique()
ct1_ids

array([10009, 10157])

In [6]:
ct1_ids_arrow = pa.array(ct1_ids, type=pa.int64())

right_optic_filter = (
    (pc.field("side") == "right") &
    pc.field("neuropil").isin(["ME", "LO", "LOP"])
)

ct1_filter = (
    pc.field("pre").isin(ct1_ids_arrow) |
    pc.field("post").isin(ct1_ids_arrow)
)

final_filter = ct1_filter & right_optic_filter

scanner = dataset.scanner(
    columns=[
        "id",
        "x", "y", "z",
        "pre", "post",
        "prepost",
        "neuropil",
        "side",
        "pre_label",
        "post_label",
        "syn_top_nt",
        "syn_top_p",
    ],
    filter=final_filter,
    batch_size=500_000,
)
ct1_synapses = scanner.to_table().to_pandas()
ct1_synapses["roi"] = ct1_synapses["neuropil"] + "_R"
print("CT1 synapses types:", ct1_synapses["prepost"].unique())
ct1_synapses.head()

CT1 synapses types: [0.]


,id,x,y,z,pre,post,prepost,neuropil,side,pre_label,post_label,syn_top_nt,syn_top_p,roi
0,1.273015e+17,24362,33541,28945,122860,10009,0.0,ME,right,dendrite,None,acetylcholine,0.994717,ME_R
1,1.276490e+17,24263,33515,29024,106853,10009,0.0,ME,right,dendrite,None,acetylcholine,0.999731,ME_R
2,1.276974e+17,24274,33559,29035,111369,10009,0.0,ME,right,dendrite,None,acetylcholine,0.997132,ME_R
3,1.277193e+17,24396,33815,29040,106853,10009,0.0,ME,right,dendrite,None,acetylcholine,0.983047,ME_R
4,1.277237e+17,24414,33792,29041,106853,10009,0.0,ME,right,dendrite,None,acetylcholine,0.996511,ME_R


In [7]:
ME["pin_neuropil"] = "ME"
LO["pin_neuropil"] = "LO"
LOP["pin_neuropil"] = "LOP"

pin_df = pd.concat([ME, LO, LOP], ignore_index=True)

pin_df = pin_df.rename(columns={
    "hex1_id": "pin_hex1_id",
    "hex2_id": "pin_hex2_id",
    "bin_depth": "pin_bin_depth",
})

pin_df = pin_df[
    [
        "pin_hex1_id",
        "pin_hex2_id",
        "pin_bin_depth",
        "x", "y", "z",
        "roi",
        "pin_neuropil",
    ]
].copy()

pin_trees = {}

for roi, sub in pin_df.groupby("roi"):
    coords = sub[["x", "y", "z"]].to_numpy()

    pin_trees[roi] = {
        "tree": cKDTree(coords),
        "pins": sub.reset_index(drop=True),
    }

print(pin_trees.keys())
pin_df

dict_keys(['LOP_R', 'LO_R', 'ME_R'])


,pin_hex1_id,pin_hex2_id,pin_bin_depth,x,y,z,roi,pin_neuropil
0,1,7,0,25358.446771,49154.658267,30777.496794,ME_R,ME
1,1,7,1,25388.072578,49110.072824,30791.128247,ME_R,ME
2,1,7,2,25418.125073,49065.873905,30804.879960,ME_R,ME
3,1,7,3,25448.589202,49022.052856,30818.747976,ME_R,ME
4,1,7,4,25479.449913,48978.601024,30832.728335,ME_R,ME
...,...,...,...,...,...,...,...,...
216094,36,35,46,19124.152423,21788.866558,40866.687726,LOP_R,LOP
216095,36,35,47,19110.098866,21816.319716,40925.316253,LOP_R,LOP
216096,36,35,48,19096.103191,21845.280559,40982.365202,LOP_R,LOP
216097,36,35,49,19082.162447,21876.490597,41039.287858,LOP_R,LOP


In [8]:
def assign_synapses_to_pins(
    synapses,
    pin_trees,
    coord_cols=("x", "y", "z"),
    roi_col="roi",
    chunk_size=100_000,
):
    """
    Assign each synapse to the nearest pin point within the same ROI.

    Adds:
        nearest_pin_distance
        pin_hex1_id
        pin_hex2_id
        pin_bin_depth
        pin_neuropil
    """

    out_chunks = []

    for roi, syn_roi in synapses.groupby(roi_col, sort=False):
        if roi not in pin_trees:
            print(f"Skipping ROI {roi}: no pin tree")
            continue

        tree = pin_trees[roi]["tree"]
        pins = pin_trees[roi]["pins"]

        n = len(syn_roi)
        print(f"Assigning {n:,} synapses in {roi}")

        for start in range(0, n, chunk_size):
            stop = min(start + chunk_size, n)

            syn_chunk = syn_roi.iloc[start:stop].copy()
            coords = syn_chunk[list(coord_cols)].to_numpy()

            distances, idx = tree.query(coords, k=1)

            matched_pins = pins.iloc[idx][[
                "pin_hex1_id",
                "pin_hex2_id",
                "pin_bin_depth",
                "pin_neuropil",
            ]].reset_index(drop=True)

            syn_chunk = syn_chunk.reset_index(drop=True)
            syn_chunk["nearest_pin_distance"] = distances

            syn_chunk = pd.concat([syn_chunk, matched_pins], axis=1)

            out_chunks.append(syn_chunk)

    if len(out_chunks) == 0:
        return pd.DataFrame()

    return pd.concat(out_chunks, ignore_index=True)

ct1_synapses_with_pins = assign_synapses_to_pins(
    ct1_synapses,
    pin_trees,
    coord_cols=("x", "y", "z"),
    roi_col="roi",
    chunk_size=300_000,
)

Assigning 100,497 synapses in ME_R
Assigning 169,456 synapses in LO_R
Assigning 16 synapses in LOP_R


In [9]:
ct1_syn = ct1_synapses_with_pins.copy()

ct1_syn["syn_hex_1"] = pd.to_numeric(ct1_syn["pin_hex1_id"], errors="coerce")
ct1_syn["syn_hex_2"] = pd.to_numeric(ct1_syn["pin_hex2_id"], errors="coerce")

ct1_syn["syn_p"] = ct1_syn["syn_hex_1"] - 18
ct1_syn["syn_q"] = ct1_syn["syn_hex_2"] - 19

ct1_syn["syn_column_id"] = (
    ct1_syn["syn_hex_1"].astype("Int64").astype(str)
    + "_"
    + ct1_syn["syn_hex_2"].astype("Int64").astype(str)
)

node_type = nodes_df[
    ["malecns_09_id", "cell_type", "side", "neurotransmitter_predicted", "neurotransmitter_score"]
].copy()

node_type["malecns_09_id"] = node_type["malecns_09_id"].astype("int64")

pre_info = node_type.rename(columns={
    "malecns_09_id": "pre",
    "cell_type": "pre_cell_type",
    "side": "pre_side",
    "neurotransmitter_predicted": "pre_nt",
    "neurotransmitter_score": "pre_nt_score",
})

post_info = node_type.rename(columns={
    "malecns_09_id": "post",
    "cell_type": "post_cell_type",
    "side": "post_side",
    "neurotransmitter_predicted": "post_nt",
    "neurotransmitter_score": "post_nt_score",
})

ct1_syn = (
    ct1_syn
    .merge(pre_info, on="pre", how="left")
    .merge(post_info, on="post", how="left")
)
ct1_syn

,id,x,y,z,pre,post,prepost,neuropil,side,pre_label,post_label,syn_top_nt,syn_top_p,roi,nearest_pin_distance,pin_hex1_id,pin_hex2_id,pin_bin_depth,pin_neuropil,syn_hex_1,syn_hex_2,syn_p,syn_q,syn_column_id,pre_cell_type,pre_side,pre_nt,pre_nt_score,post_cell_type,post_side,post_nt,post_nt_score
0,1.273015e+17,24362,33541,28945,122860,10009,0.0,ME,right,dendrite,None,acetylcholine,0.994717,ME_R,153.794018,10,28,112,ME,10,28,-8,9,10_28,T4c,right,acetylcholine,0.941011,CT1,left,gaba,0.860479
1,1.276490e+17,24263,33515,29024,106853,10009,0.0,ME,right,dendrite,None,acetylcholine,0.999731,ME_R,199.678217,10,28,113,ME,10,28,-8,9,10_28,T4a,right,acetylcholine,0.974427,CT1,left,gaba,0.860479
2,1.276974e+17,24274,33559,29035,111369,10009,0.0,ME,right,dendrite,None,acetylcholine,0.997132,ME_R,157.767225,10,28,113,ME,10,28,-8,9,10_28,T4a,right,acetylcholine,0.974427,CT1,left,gaba,0.860479
3,1.277193e+17,24396,33815,29040,106853,10009,0.0,ME,right,dendrite,None,acetylcholine,0.983047,ME_R,139.054867,10,28,113,ME,10,28,-8,9,10_28,T4a,right,acetylcholine,0.974427,CT1,left,gaba,0.860479
4,1.277237e+17,24414,33792,29041,106853,10009,0.0,ME,right,dendrite,None,acetylcholine,0.996511,ME_R,120.301827,10,28,113,ME,10,28,-8,9,10_28,T4a,right,acetylcholine,0.974427,CT1,left,gaba,0.860479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269964,1.745629e+17,30830,36786,39691,10009,732676592,0.0,LOP,right,dendrite,None,serotonin,0.461294,LOP_R,1051.682009,5,19,1,LOP,5,19,-13,0,5_19,CT1,left,gaba,0.860479,NaN,NaN,NaN,NaN
269965,1.745629e+17,30830,36786,39691,10009,336833270,0.0,LOP,right,dendrite,None,serotonin,0.461294,LOP_R,1051.682009,5,19,1,LOP,5,19,-13,0,5_19,CT1,left,gaba,0.860479,NaN,NaN,NaN,NaN
269966,1.745629e+17,30830,36786,39691,10009,456250985,0.0,LOP,right,dendrite,None,serotonin,0.461294,LOP_R,1051.682009,5,19,1,LOP,5,19,-13,0,5_19,CT1,left,gaba,0.860479,NaN,NaN,NaN,NaN
269967,1.745629e+17,30830,36786,39691,10009,314362958,0.0,LOP,right,dendrite,None,serotonin,0.461294,LOP_R,1051.682009,5,19,1,LOP,5,19,-13,0,5_19,CT1,left,gaba,0.860479,NaN,NaN,NaN,NaN


In [13]:
# group_cols = [
#     "pre",
#     "post",
#     "neuropil",
#     "roi",
#     "syn_column_id",
#     "syn_hex_1",
#     "syn_hex_2",
#     "syn_p",
#     "syn_q",
#     "pin_neuropil",
# ]

group_cols = ["pre", "post", "pre_cell_type", 
              "post_cell_type", "neuropil", "roi", 
              "syn_column_id", 
              "syn_hex_1", "syn_hex_2", "syn_p", "syn_q"]
ct1_edges = (
    ct1_syn
    .groupby(group_cols, dropna=False)
    .agg(
        # pre_cell_type=("pre_cell_type", "first"),
        # post_cell_type=("post_cell_type", "first"),
        pre_side=("pre_side", "first"),
        post_side=("post_side", "first"),
        count=("id", "size"),  # or nunique if you want unique IDs only
        mean_x=("x", "mean"),
        mean_y=("y", "mean"),
        mean_z=("z", "mean"),
        mean_pin_depth=("pin_bin_depth", "mean"),
        min_pin_depth=("pin_bin_depth", "min"),
        max_pin_depth=("pin_bin_depth", "max"),
        mean_pin_dist=("nearest_pin_distance", "mean"),
        median_pin_dist=("nearest_pin_distance", "median"),
        syn_top_nt=("syn_top_nt", lambda x: x.mode().iloc[0] if len(x.mode()) else pd.NA),
        mean_syn_top_p=("syn_top_p", "mean"),
    )
    .reset_index()
)
ct1_edges.drop(columns=["roi", "mean_x", "mean_y", "mean_z", "syn_column_id", "median_pin_dist", "pre_side", "post_side",
                        "mean_pin_depth",  "min_pin_depth", "max_pin_depth", "mean_pin_dist"], inplace=True)

ct1_edges = ct1_edges.dropna(
    subset=["pre_cell_type", "post_cell_type"]
).reset_index(drop=True)

print(ct1_edges["count"].min())

np.int64(1)

In [14]:
ct1_edges.to_feather(out_dir/"ct1_edgelist_full.feather")

In [15]:
print("Raw CT1 synapse rows:", len(ct1_syn))
print("Aggregated CT1 edge rows:", len(ct1_edges))
print("Total synapses preserved:", ct1_edges["count"].sum())
print("Unique CT1 pairs:", ct1_syn[["pre", "post"]].drop_duplicates().shape[0])

Raw CT1 synapse rows: 269969
Aggregated CT1 edge rows: 55895
Total synapses preserved: 220424
Unique CT1 pairs: 55950
